In [2]:
%pip install pyarrow


   ---------------------------------------- 0.0/27.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/27.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/27.9 MB ? eta -:--:--
    --------------------------------------- 0.5/27.9 MB 1.2 MB/s eta 0:00:24
   - -------------------------------------- 0.8/27.9 MB 1.2 MB/s eta 0:00:24
   - -------------------------------------- 1.0/27.9 MB 1.2 MB/s eta 0:00:24
   - -------------------------------------- 1.3/27.9 MB 1.2 MB/s eta 0:00:23
   -- ------------------------------------- 1.6/27.9 MB 1.2 MB/s eta 0:00:23
   -- ------------------------------------- 1.8/27.9 MB 1.2 MB/s eta 0:00:23
   --- ------------------------------------ 2.1/27.9 MB 1.2 MB/s eta 0:00:23
   --- ------------------------------------ 2.4/27.9 MB 1.2 MB/s eta 0:00:22
   --- ------------------------------------ 2.4/27.9 MB 1.2 MB/s eta 0:00:22
   --- ------------------------------------ 2.6/27.9 MB 1.2 MB/s eta 0:00:22
   ---- ------------


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
%pip install --upgrade --force-reinstall pyarrow



  Using cached pyarrow-25.0.1-cp313-cp313-win_amd64.whl.metadata (3.0 kB)
Using cached pyarrow-25.0.1-cp313-cp313-win_amd64.whl (27.9 MB)
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 25.0.1
    Uninstalling pyarrow-25.0.1:
      Successfully uninstalled pyarrow-25.0.1
Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os

input_file = r"C:\Users\adiz\Downloads\archive(1)\credit_card_transactions-ibm_v2.csv"
output_dir = r"C:\Users\adiz\Downloads\archive(1)\credit_card_transactions_parquet_7"

os.makedirs(output_dir, exist_ok=True)

chunksize = 500_000
num_files = 7

writers = [None] * num_files
row_counts = [0] * num_files

try:

    for i, chunk in enumerate(
        pd.read_csv(input_file, chunksize=chunksize)
    ):

        # Distribute chunks across the 7 files
        file_index = i % num_files

        output_file = os.path.join(
            output_dir,
            f"transactions_{file_index:02d}.parquet"
        )

        # Convert pandas DataFrame to Arrow table
        table = pa.Table.from_pandas(
            chunk,
            preserve_index=False
        )

        # Create writer for this file on first use
        if writers[file_index] is None:

            writers[file_index] = pq.ParquetWriter(
                output_file,
                table.schema,
                compression="snappy"
            )

        # Write chunk
        writers[file_index].write_table(table)

        row_counts[file_index] += len(chunk)

        print(
            f"Processed chunk {i + 1} "
            f"→ file {file_index:02d} "
            f"({len(chunk):,} rows)"
        )

finally:

    # Always close the writers
    for writer in writers:
        if writer is not None:
            writer.close()


print("\nConversion complete!")

for i, count in enumerate(row_counts):
    print(
        f"transactions_{i:02d}.parquet: "
        f"{count:,} rows"
    )


In [ ]:
"""
Check conversion:
- Exactly 7 files
- All files readable
- Same row count
- Same columns/order
- Consistent Parquet schema
- No leading zeros in original Merchant Name
- Total size < 2 GB
"""

import os
import glob
import pandas as pd
import pyarrow.parquet as pq

input_file = r"C:\Users\adiz\Downloads\archive(1)\credit_card_transactions-ibm_v2.csv"
output_dir = r"C:\Users\adiz\Downloads\archive(1)\credit_card_transactions_parquet_7"
EXPECTED_PARQUET_FILES = 7
MAX_SIZE_GB = 2
CHUNK_SIZE = 500_000

results = {}

def record_result(name, passed):
    results[name] = passed
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

print("\n=== INPUT CSV CHECK ===")
record_result("CSV file exists", os.path.exists(input_file))

parquet_files = sorted(glob.glob(os.path.join(output_dir, "*.parquet")))
print(f"\n=== PARQUET FILE CHECK ===")
print(f"Parquet files found: {len(parquet_files)}")
for file in parquet_files:
    print(f"  {os.path.basename(file)}")
record_result(f"Exactly {EXPECTED_PARQUET_FILES} Parquet files", len(parquet_files) == EXPECTED_PARQUET_FILES)

print("\n=== PARQUET READABILITY CHECK ===")
all_readable = True
for file in parquet_files:
    try:
        pf = pq.ParquetFile(file)
        print(f"PASS: {os.path.basename(file)} ({pf.metadata.num_rows:,} rows, {len(pf.schema_arrow.names)} columns)")
    except Exception as e:
        all_readable = False
        print(f"FAIL: {os.path.basename(file)} - {e}")
record_result("All Parquet files are readable", all_readable)

print("\n=== COLUMN CHECK ===")
csv_columns = pd.read_csv(input_file, nrows=0).columns.tolist()
all_columns_match = True
for file in parquet_files:
    parquet_columns = pq.read_schema(file).names
    if parquet_columns == csv_columns:
        print(f"PASS: {os.path.basename(file)} columns match CSV")
    else:
        all_columns_match = False
        print(f"FAIL: {os.path.basename(file)} columns do not match CSV")
record_result("Column names and order match", all_columns_match)

print("\n=== ROW COUNT CHECK ===")
csv_rows = sum(len(chunk) for chunk in pd.read_csv(input_file, chunksize=CHUNK_SIZE))
parquet_rows = 0
for file in parquet_files:
    rows = pq.read_metadata(file).num_rows
    parquet_rows += rows
    print(f"{os.path.basename(file)}: {rows:,} rows")
print(f"CSV rows: {csv_rows:,}")
print(f"Parquet rows: {parquet_rows:,}")
record_result("CSV and Parquet row counts match", csv_rows == parquet_rows)

print("\n=== MERCHANT NAME LEADING ZERO CHECK ===")
leading_zero_count = 0
merchant_total = 0
examples = []
for chunk in pd.read_csv(
    input_file,
    usecols=["Merchant Name"],
    dtype={"Merchant Name": "string"},
    chunksize=CHUNK_SIZE
):
    merchant = chunk["Merchant Name"]
    merchant_total += len(merchant)
    bad = merchant.str.startswith("0", na=False)
    count = int(bad.sum())
    leading_zero_count += count
    if count and len(examples) < 20:
        examples.extend(merchant[bad].head(20 - len(examples)).tolist())

print(f"Merchant Name values checked: {merchant_total:,}")
print(f"Leading-zero values: {leading_zero_count:,}")
if examples:
    print(f"Examples: {examples}")
record_result("Merchant Name has no leading zeros", leading_zero_count == 0)

print("\n=== PARQUET SCHEMA CHECK ===")
schema_consistent = True
first_schema = None
for file in parquet_files:
    schema = {field.name: str(field.type) for field in pq.read_schema(file)}
    if first_schema is None:
        first_schema = schema
    elif schema != first_schema:
        schema_consistent = False
    print(f"{os.path.basename(file)}: schema checked")
record_result("All Parquet files have consistent schemas", schema_consistent)

print("\n=== MERCHANT NAME DATA TYPE ===")
merchant_type = first_schema.get("Merchant Name", "NOT FOUND")
print(f"Merchant Name Parquet type: {merchant_type}")
if merchant_type == "int64":
    print("INFO: Merchant Name is stored as int64.")
    print("The original CSV was checked as string for leading zeros.")

print("\n=== NULL VALUE CHECK ===")
parquet_null_counts = {}
for file in parquet_files:
    df = pd.read_parquet(file)
    null_counts = df.isna().sum()
    for column, count in null_counts.items():
        parquet_null_counts[column] = parquet_null_counts.get(column, 0) + int(count)

print("Total Parquet null counts:")
for column, count in parquet_null_counts.items():
    if count > 0:
        print(f"  {column}: {count:,}")
record_result("Null count inspection completed", True)

print("\n=== DUPLICATE ROW CHECK ===")
total_duplicates = 0
for file in parquet_files:
    df = pd.read_parquet(file)
    duplicates = int(df.duplicated().sum())
    total_duplicates += duplicates
    print(f"{os.path.basename(file)}: {duplicates:,} duplicate rows")
print(f"Total duplicate rows: {total_duplicates:,}")
print("NOTE: Duplicate rows are not automatically an error.")

print("\n=== NUMERIC RANGE CHECK ===")
numeric_ranges = {}
for file in parquet_files:
    df = pd.read_parquet(file)
    for column in df.select_dtypes(include="number").columns:
        min_value = df[column].min()
        max_value = df[column].max()
        if column not in numeric_ranges:
            numeric_ranges[column] = {"min": min_value, "max": max_value}
        else:
            numeric_ranges[column]["min"] = min(numeric_ranges[column]["min"], min_value)
            numeric_ranges[column]["max"] = max(numeric_ranges[column]["max"], max_value)

for column, values in numeric_ranges.items():
    print(f"{column}: min={values['min']}, max={values['max']}")

print("\n=== SIZE CHECK ===")
total_size = sum(os.path.getsize(file) for file in parquet_files)
size_mb = total_size / (1024 ** 2)
size_gb = total_size / (1024 ** 3)
print(f"Total Parquet size: {size_mb:.2f} MB ({size_gb:.3f} GB)")
record_result(f"Total Parquet size is below {MAX_SIZE_GB} GB", size_gb < MAX_SIZE_GB)

print("\n=== ROW DISTRIBUTION ===")
for file in parquet_files:
    rows = pq.read_metadata(file).num_rows
    percentage = rows / parquet_rows * 100 if parquet_rows else 0
    print(f"{os.path.basename(file)}: {rows:,} rows ({percentage:.2f}%)")

print("\n=== FINAL RESULT ===")
core_checks = [
    results.get(f"Exactly {EXPECTED_PARQUET_FILES} Parquet files", False),
    results.get("CSV and Parquet row counts match", False),
    results.get("Column names and order match", False),
    results.get("All Parquet files are readable", False),
    results.get("All Parquet files have consistent schemas", False),
    results.get("Merchant Name has no leading zeros", False),
    results.get(f"Total Parquet size is below {MAX_SIZE_GB} GB", False)
]

for name, passed in results.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

print("\nOVERALL RESULT:", "PASS" if all(core_checks) else "FAIL")
